# MPC vs LQR vs Robust Control!/usr/bin/env python3

**Figures generated:** mpc_comparison.svgRun all cells below to regenerate.

In [ ]:
%matplotlib inline

In [ ]:
"""LQR vs Robust vs MPC under a changing disturbance===================================================First-order plant with input disturbance, controlled at 10 Hz.MPC uses a 2-second preview of the known disturbance profile."""import numpy as npfrom scipy.linalg import solve_discrete_arefrom scipy.optimize import minimizeimport matplotlibimport matplotlib.pyplot as plt# ==================================================================matplotlib.rcParams.update({    "svg.fonttype": "path", "mathtext.fontset": "cm",    "font.family": "serif", "font.size": 10,    "axes.labelsize": 11, "axes.titlesize": 11,    "legend.fontsize": 8.5,    "xtick.labelsize": 9, "ytick.labelsize": 9,    "lines.linewidth": 1.5, "axes.linewidth": 0.8,    "xtick.major.width": 0.6, "ytick.major.width": 0.6,    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,})BLUE   = (0.0000, 0.4470, 0.7410)ORANGE = (0.8500, 0.3250, 0.0980)GREEN  = (0.4660, 0.6740, 0.1880)GRAY   = (0.50, 0.50, 0.50)LGRAY  = (0.88, 0.88, 0.88)RED_L  = (1.0, 0.86, 0.86)# ==================================================================#  Plant:  y_dot = -a y + b (u + d)# ==================================================================a_pl, b_pl = 1.0, 2.0dt = 0.1                            # 10 Hz control# Exact ZOH discretisation  (scalar)Ad = np.exp(-a_pl * dt)Bd = b_pl / a_pl * (1 - Ad)         # from ∫₀^dt e^{-a τ} b dτ# state: x = y (scalar),  x_{k+1} = Ad x_k + Bd (u_k + d_k)# Augmented state [x, x_i] with integral x_i for offset-free tracking# x_i,k+1 = x_i,k + dt * x_kA_aug = np.array([[Ad,   0.0],                  [dt*Ad, 1.0]])B_aug = np.array([[Bd],                  [dt*Bd]])nx_aug = 2# ==================================================================#  Constraints# ==================================================================u_max  = 2.0          # tight input saturation — exactly at d_maxy_safe = 0.5          # output safe-set half-width# ==================================================================#  Disturbance# ==================================================================T_total = 18.0N_sim   = int(T_total / dt)t_sim   = np.arange(N_sim) * dtdef disturbance(tv):    if   tv < 2:  return  0.0    elif tv < 6:  return  2.0    elif tv < 10: return -1.5    elif tv < 14: return  1.0    else:         return  0.0d_vec = np.array([disturbance(ti) for ti in t_sim])# ==================================================================#  LQR with integral (aggressive)# ==================================================================Q_lqr = np.diag([50.0, 200.0])       # heavy on integral → fast rejectionR_lqr = np.array([[0.5]])P_l   = solve_discrete_are(A_aug, B_aug, Q_lqr, R_lqr)K_lqr = np.linalg.solve(R_lqr + B_aug.T @ P_l @ B_aug,                         B_aug.T @ P_l @ A_aug)           # 1×2# ==================================================================#  Robust (conservative: heavier R)# ==================================================================Q_rob = np.diag([20.0, 60.0])R_rob = np.array([[5.0]])             # 10× heavierP_r   = solve_discrete_are(A_aug, B_aug, Q_rob, R_rob)K_rob = np.linalg.solve(R_rob + B_aug.T @ P_r @ B_aug,                         B_aug.T @ P_r @ A_aug)print(f"LQR gains: {K_lqr[0]}")print(f"Rob gains: {K_rob[0]}")# ==================================================================#  MPC  (scalar state, with disturbance preview, box constraints)# ==================================================================N_hor = 20                            # 2 s previewQ_m   = 50.0                         # same weight on y as LQRR_m   = 0.5Q_ter = 5 * Q_m                      # terminal cost# Prediction matrices (scalar plant, vector U)Phi     = np.zeros(N_hor)             # x_i = Ad^i x0 + …Gamma   = np.zeros((N_hor, N_hor))Gamma_d = np.zeros((N_hor, N_hor))for i in range(N_hor):    Phi[i] = Ad**(i+1)    for j in range(i+1):        Gamma[i, j]   = Ad**(i-j) * Bd        Gamma_d[i, j] = Ad**(i-j) * Bd# Cost:  J = || Phi x0 + Gamma U + Gamma_d D ||^2_Q  +  ||U||^2_R  (+ terminal)Qvec = np.full(N_hor, Q_m); Qvec[-1] = Q_terQ_bar = np.diag(Qvec)R_bar = R_m * np.eye(N_hor)H_mpc = Gamma.T @ Q_bar @ Gamma + R_barH_mpc = 0.5 * (H_mpc + H_mpc.T)GtQ   = Gamma.T @ Q_barbounds_mpc = [(-u_max, u_max)] * N_horU_warm = np.zeros(N_hor)def mpc_step(x0, d_preview):    global U_warm    D = d_preview.copy()    if len(D) < N_hor:        D = np.concatenate([D, np.full(N_hor - len(D), D[-1])])    D = D[:N_hor]    rhs = Phi * x0 + Gamma_d @ D    f   = GtQ @ rhs    res = minimize(lambda U: 0.5 * U @ H_mpc @ U + f @ U,                   U_warm, jac=lambda U: H_mpc @ U + f,                   bounds=bounds_mpc, method="L-BFGS-B",                   options={"maxiter": 60, "ftol": 1e-12})    U_warm = np.roll(res.x, -1); U_warm[-1] = res.x[-1]    return res.x[0]# ==================================================================#  Simulation# ==================================================================def simulate(ctrl_fn):    x    = 0.0                       # plant state (scalar)    x_i  = 0.0                       # integral state    yh   = np.zeros(N_sim)    uh   = np.zeros(N_sim)    for k in range(N_sim):        yh[k] = x        u = ctrl_fn(x, x_i, k)        u = np.clip(u, -u_max, u_max)        uh[k] = u        x = Ad * x + Bd * (u + d_vec[k])        x_i += dt * x    return yh, uhctrl_lqr_fn = lambda x, xi, k: -(K_lqr[0,0]*x + K_lqr[0,1]*xi)ctrl_rob_fn = lambda x, xi, k: -(K_rob[0,0]*x + K_rob[0,1]*xi)def ctrl_mpc_fn(x, xi, k):    dp = d_vec[k:k+N_hor]    if len(dp) == 0: dp = np.array([0.0])    return mpc_step(x, dp)print("LQR  …", end=" ", flush=True)y_lqr, u_lqr = simulate(ctrl_lqr_fn); print("ok")print("Rob  …", end=" ", flush=True)y_rob, u_rob = simulate(ctrl_rob_fn); print("ok")print("MPC  …", end=" ", flush=True)y_mpc, u_mpc = simulate(ctrl_mpc_fn); print("ok\n")for tag, yy in [("LQR",y_lqr),("Robust",y_rob),("MPC",y_mpc)]:    pk = np.max(np.abs(yy))    vt = np.sum(np.abs(yy)>y_safe)*dt    print(f"  {tag:7s}: peak |y|={pk:.3f}, outside safe set {vt:.1f} s")# ==================================================================

In [ ]:
#  Figure# ==================================================================fig, axes = plt.subplots(3, 1, figsize=(9.5, 7.2), sharex=True,                         gridspec_kw={"height_ratios": [0.7, 1, 1]})fig.subplots_adjust(left=0.09, right=0.97, bottom=0.07, top=0.87,                    hspace=0.12)fig.suptitle("Disturbance rejection: LQR vs. robust control vs. MPC",             fontsize=13, fontweight="bold", y=0.97)# ---- shared legend (between title and top panel) ----from matplotlib.lines import Line2Dfrom matplotlib.patches import Patchleg = [    Line2D([0],[0], color=BLUE,  lw=1.8, label="LQR (state feedback + integral)"),    Line2D([0],[0], color=ORANGE,lw=1.8,           label=r"Robust ($H_\infty$-type, conservative)"),    Line2D([0],[0], color=GREEN, lw=1.8, label="MPC (2 s disturbance preview)"),    Patch(facecolor=RED_L, edgecolor="red", lw=0.6, ls="--",          label=r"Unsafe region ($|y|>y_{\max}$)"),]fig.legend(handles=leg, loc="upper center",           bbox_to_anchor=(0.53, 0.935), ncol=2,           fontsize=8.5, framealpha=0.95, edgecolor="0.82",           columnspacing=1.4, handlelength=1.8)# ---- Panel 0: disturbance ----ax0 = axes[0]ax0.fill_between(t_sim, d_vec, step="post", color=LGRAY, edgecolor=GRAY,                 lw=0.8)ax0.set_ylabel(r"$d(t)$")ax0.set_ylim(-2.5, 3.2)ax0.text(0.99, 0.90, "known disturbance profile",         transform=ax0.transAxes, ha="right", va="top",         fontsize=8.5, fontstyle="italic", color="0.45")# ---- Panel 1: output ----ax1 = axes[1]ym = max(np.max(np.abs(y_rob)), np.max(np.abs(y_lqr)), y_safe) * 1.25ax1.axhspan( y_safe,  ym, color=RED_L, alpha=0.45, zorder=0)ax1.axhspan(-ym, -y_safe,  color=RED_L, alpha=0.45, zorder=0)ax1.axhline( y_safe, color="red", ls="--", lw=0.8, alpha=0.7)ax1.axhline(-y_safe, color="red", ls="--", lw=0.8, alpha=0.7)ax1.axhline(0, color="k", lw=0.4)ax1.plot(t_sim, y_lqr, color=BLUE)ax1.plot(t_sim, y_rob, color=ORANGE)ax1.plot(t_sim, y_mpc, color=GREEN)ax1.set_ylabel(r"Output $y(t)$")ax1.set_ylim(-ym, ym)ax1.text(17.7, y_safe+0.02, r"$y_{\max}$", color="red",         fontsize=7.5, ha="right", va="bottom")ax1.text(17.7, -y_safe-0.02, r"$-y_{\max}$", color="red",         fontsize=7.5, ha="right", va="top")# ---- Panel 2: control ----ax2 = axes[2]ax2.axhline(0, color="k", lw=0.4)um_plot = u_max * 1.30ax2.axhspan( u_max, um_plot, color=RED_L, alpha=0.45, zorder=0)ax2.axhspan(-um_plot, -u_max, color=RED_L, alpha=0.45, zorder=0)ax2.axhline( u_max, color="red", ls="--", lw=0.8, alpha=0.7)ax2.axhline(-u_max, color="red", ls="--", lw=0.8, alpha=0.7)ax2.plot(t_sim, u_lqr, color=BLUE)ax2.plot(t_sim, u_rob, color=ORANGE)ax2.plot(t_sim, u_mpc, color=GREEN)ax2.set_ylabel(r"Control $u(t)$")ax2.set_xlabel(r"Time $t$ [s]")ax2.set_ylim(-um_plot, um_plot)ax2.text(17.7,  u_max+0.03, r"$u_{\max}$", color="red",         fontsize=7.5, ha="right", va="bottom")ax2.text(17.7, -u_max-0.03, r"$-u_{\max}$", color="red",         fontsize=7.5, ha="right", va="top")for ax in axes:    ax.set_xlim(0, T_total)# preview-window shadingfor ts in [2, 6, 10, 14]:    for ax in axes[1:]:        ax.axvspan(max(0, ts - N_hor*dt), ts,                   color=GREEN, alpha=0.04, zorder=0)fig.savefig("mpc_comparison.svg", bbox_inches="tight")fig.savefig("mpc_comparison.png", dpi=200, bbox_inches="tight")print("\nFigure saved.")plt.close(fig)